# Encoder output inspection — nominal vs learned-flip tagger

Loads a **fixed, already-trained** `transformer_jet_classifier_auto_scaler.py` checkpoint (frozen backbone + learned `flip_scale`) and lets you inspect the transformer's encoder output (the CLS embedding, i.e. `h[:, 0]`) for the nominal vs flipped view of the same jets.

Workflow:
1. Run the **Config**, **Data loading & model definition**, and **Load model + data** / **Run inference** cells once.
2. Re-run any of the plotting cells as many times as you like — they all read from the `h_nom` / `h_flip` / `p_nom` / `p_flip` / `y` arrays produced by the inference cell.

In [ ]:
import hashlib
import json
import os

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

## Config

Point `RUN_DIR` at a completed `transformer_jet_classifier_auto_scaler.py` output directory. Everything else (architecture, track fields, data file, cache dir, checkpoint name) is read straight from that run's saved `config.json`, so there's no risk of it drifting out of sync with how the model was actually trained.

In [ ]:
RUN_DIR = "./transformer_results_auto_scaler/"
N_PLOT_SAMPLE = 4000  # subsample size for scatter/PCA plots (histograms use the full set)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open(os.path.join(RUN_DIR, "config.json")) as _f:
    run_cfg = json.load(_f)

TRAIN_FILE       = run_cfg["train_file"]
N_TEST           = run_cfg["n_test"]
TOP_K            = run_cfg["top_k"]
CACHE_DIR        = run_cfg["train_cache_dir"]
TRACK_FIELDS     = run_cfg["track_fields"]
FLAVOUR_TO_LABEL = {int(k): v for k, v in run_cfg["flavour_to_label"].items()}
CLASS_NAMES      = run_cfg["class_names"]
COLOURS          = run_cfg["colours"]
D_MODEL          = run_cfg["d_model"]
N_HEADS          = run_cfg["n_heads"]
N_LAYERS         = run_cfg["n_layers"]
D_FFN            = run_cfg["d_ffn"]
DROPOUT          = run_cfg["dropout"]
N_ORIGINS        = run_cfg["n_origins"]
N_FEATS          = len(TRACK_FIELDS)
CHECKPOINT_PATH  = os.path.join(RUN_DIR, run_cfg["model_name"])

PLOT_DIR = os.path.join(RUN_DIR, "encoder_inspection/")
os.makedirs(PLOT_DIR, exist_ok=True)

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Data file:  {TRAIN_FILE}")
print(f"Track fields ({N_FEATS}): {TRACK_FIELDS}")

## Data loading & model definition

In [ ]:
# Standalone copy of load_tracks from transformer_jet_classifier_auto_scaler.py.
# Keyed only by the index array, so if CACHE_DIR is the same as the training run's
# train_cache_dir, the held-out test set reconstructed below will hit that run's
# existing cache instead of recomputing from the H5 file.
def _cache_key(idx):
    h = hashlib.md5(idx.tobytes()).hexdigest()[:12]
    return os.path.join(CACHE_DIR, f"tracks_{h}.npz")


def load_tracks(path, idx):
    """Returns (N, K, F) features, (N, K) validity mask, (N,) labels, (N, K) origins."""
    cp = _cache_key(idx)
    if os.path.exists(cp):
        d = np.load(cp)
        if "origins" in d:
            return d["X"], d["mask"], d["y"], d["origins"]

    with h5py.File(path, "r") as f:
        flavour_id = f["jets"]["HadronConeExclTruthLabelID"][idx]
        keep_jet   = np.isin(flavour_id, list(FLAVOUR_TO_LABEL.keys()))
        fidx       = idx[keep_jet]

        valid  = f["tracks"]["valid"][fidx]
        d0     = f["tracks"]["d0"][fidx].astype(np.float32)
        ip2d   = f["tracks"]["lifetimeSignedD0Significance"][fidx].astype(np.float32)
        origin = f["tracks"]["GN2v01_trackOrigin"][fidx].astype(np.int8)
        arrs   = {fld: f["tracks"][fld][fidx].astype(np.float32) for fld in TRACK_FIELDS}

    keep = valid & (np.abs(d0) < 3.5)

    sort_key = ip2d.copy()
    sort_key[~keep] = -np.inf
    order = np.argsort(-sort_key, axis=1)

    feat_list = [arrs[fld] for fld in TRACK_FIELDS]
    feats = np.stack(feat_list, axis=-1)

    topk_idx    = order[:, :TOP_K]
    rows        = np.arange(len(fidx))[:, None]
    topk_feat   = feats[rows, topk_idx]
    topk_valid  = keep[rows, topk_idx]
    topk_feat   = np.where(topk_valid[:, :, None], topk_feat, 0.0).astype(np.float32)
    topk_origin = origin[rows, topk_idx].astype(np.int64)
    topk_origin[~topk_valid] = -1

    labels = np.array([FLAVOUR_TO_LABEL[v] for v in flavour_id[keep_jet]], dtype=np.int64)

    np.savez(cp, X=topk_feat, mask=topk_valid, y=labels, origins=topk_origin)
    return topk_feat, topk_valid, labels, topk_origin

In [ ]:
# Verbatim copy of the JetTransformer architecture from
# transformer_jet_classifier_auto_scaler.py — must match exactly so the checkpoint's
# state_dict (backbone weights + flip_scale + flip_mask, all saved together) loads cleanly.
class JetTransformer(nn.Module):
    def __init__(self, in_dim, d_model, n_heads, n_layers, d_ffn, dropout,
                 n_classes, n_origins, flip_scale_init_std=0.05, flippable_mask=None):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, d_model)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        if flippable_mask is None:
            flippable_mask = [True] * in_dim
        self.register_buffer("flip_mask", torch.tensor(flippable_mask, dtype=torch.bool))
        flip_init = torch.ones(in_dim) + flip_scale_init_std * torch.randn(in_dim)
        flip_init = torch.where(self.flip_mask, flip_init, torch.ones(in_dim))
        self.flip_scale = nn.Parameter(flip_init)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ffn,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder     = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier  = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, n_classes),
        )
        self.origin_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, n_origins),
        )

    def forward(self, x, mask, flip=False):
        B = x.size(0)
        if flip:
            scale = torch.where(self.flip_mask, self.flip_scale, torch.ones_like(self.flip_scale))
            x = x * scale
        h = self.input_proj(x)

        cls = self.cls_token.expand(B, -1, -1)
        h   = torch.cat([cls, h], dim=1)

        cls_valid            = torch.ones(B, 1, dtype=torch.bool, device=x.device)
        src_key_padding_mask = ~torch.cat([cls_valid, mask], dim=1)

        h = self.encoder(h, src_key_padding_mask=src_key_padding_mask)
        # h[:, 0]  -> CLS token  -> jet classification / encoder output we're inspecting
        # h[:, 1:] -> track tokens -> per-track origin classification
        return self.classifier(h[:, 0]), self.origin_head(h[:, 1:]), h[:, 0]

## Load model + data

In [ ]:
model = JetTransformer(N_FEATS, D_MODEL, N_HEADS, N_LAYERS, D_FFN, DROPOUT,
                       n_classes=3, n_origins=N_ORIGINS).to(DEVICE)
state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:    print(f"missing keys: {missing}")
if unexpected: print(f"unexpected keys: {unexpected}")
model.eval()
print(f"Loaded {CHECKPOINT_PATH}")
print(f"Flippable fields: {[fld for fld, m in zip(TRACK_FIELDS, model.flip_mask.tolist()) if m]}")

In [ ]:
# Reconstruct the same held-out test set the training run validated on (same seed +
# same n_test), so a cache hit against CACHE_DIR is likely if it's the training run's own.
rng = np.random.default_rng(42)
with h5py.File(TRAIN_FILE, "r") as f:
    all_flavours = f["jets"]["HadronConeExclTruthLabelID"][:]
valid_mask = np.isin(all_flavours, list(FLAVOUR_TO_LABEL.keys()))
valid_idx  = rng.permutation(np.where(valid_mask)[0])
test_idx   = np.sort(valid_idx[-N_TEST:])

X_test, mask_test, y_test, origins_test = load_tracks(TRAIN_FILE, test_idx)
X_test    = torch.from_numpy(X_test)
mask_test = torch.from_numpy(mask_test)
print(f"Loaded {len(y_test):,} test jets — "
      f"b:{(y_test==0).sum():,}  c:{(y_test==1).sum():,}  light:{(y_test==2).sum():,}")

## Run inference (nominal vs flipped)

Produces, for every jet in the test set: `h_nom`/`h_flip` (the 32-dim CLS embedding — the encoder output), `p_nom`/`p_flip` (softmax class probabilities), and `y` (true label).

In [ ]:
BATCH = 1024
h_nom_list, h_flip_list, p_nom_list, p_flip_list = [], [], [], []
with torch.no_grad():
    for i in range(0, len(y_test), BATCH):
        X_b    = X_test[i:i+BATCH].to(DEVICE)
        mask_b = mask_test[i:i+BATCH].to(DEVICE)
        logits_nom,  _, h_nom_b  = model(X_b, mask_b, flip=False)
        logits_flip, _, h_flip_b = model(X_b, mask_b, flip=True)
        h_nom_list.append(h_nom_b.cpu())
        h_flip_list.append(h_flip_b.cpu())
        p_nom_list.append(torch.softmax(logits_nom, dim=1).cpu())
        p_flip_list.append(torch.softmax(logits_flip, dim=1).cpu())

h_nom  = torch.cat(h_nom_list).numpy()   # (N, d_model)
h_flip = torch.cat(h_flip_list).numpy()  # (N, d_model)
p_nom  = torch.cat(p_nom_list).numpy()   # (N, 3)
p_flip = torch.cat(p_flip_list).numpy()  # (N, 3)
y      = y_test
print(f"h_nom {h_nom.shape}, h_flip {h_flip.shape}")

## Plot: cosine similarity of encoder output, nominal vs flipped

The core check — per jet, how similar is `h_flip` to `h_nom`? Split by true flavour: light should cluster near 1 (invariant), b should be pulled away from 1 (sensitive to flip).

In [ ]:
cos_sim = np.sum(h_nom * h_flip, axis=1) / (
    np.linalg.norm(h_nom, axis=1) * np.linalg.norm(h_flip, axis=1) + 1e-10)

fig, ax = plt.subplots(figsize=(7, 5))
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = y == cls_idx
    ax.hist(cos_sim[m], bins=60, range=(-1, 1), histtype="step",
            label=cls_name, color=COLOURS[cls_name], linewidth=2, density=True)
ax.set_xlabel("cos_sim(h_nom, h_flip)"); ax.set_ylabel("Density")
ax.set_title("Encoder output similarity, nominal vs flipped")
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR + "cos_sim_by_flavour.png", dpi=150, bbox_inches="tight")
print("Saved cos_sim_by_flavour.png")
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = y == cls_idx
    print(f"  {cls_name:10s} mean cos_sim = {cos_sim[m].mean():+.4f}")

## Plot: 2D PCA of encoder output

Fits PCA jointly on nominal + flipped embeddings (shared basis), then shows both as a scatter, colored by true flavour. A random subsample is connected nom -> flip with a thin line so you can see how far and in what direction the flip moves each jet's embedding.

In [ ]:
rng_plot = np.random.default_rng(0)
sel = rng_plot.choice(len(y), size=min(N_PLOT_SAMPLE, len(y)), replace=False)

pca = PCA(n_components=2)
pca.fit(np.concatenate([h_nom, h_flip], axis=0))
nom_2d  = pca.transform(h_nom[sel])
flip_2d = pca.transform(h_flip[sel])
y_sel   = y[sel]

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = y_sel == cls_idx
    axes[0].scatter(nom_2d[m, 0], nom_2d[m, 1], s=4, alpha=0.4,
                    color=COLOURS[cls_name], label=cls_name, linewidths=0)
    axes[1].scatter(flip_2d[m, 0], flip_2d[m, 1], s=4, alpha=0.4,
                    color=COLOURS[cls_name], label=cls_name, linewidths=0)
axes[0].set_title("Nominal"); axes[1].set_title("Flipped")
for ax in axes:
    ax.set_xlabel("PC1"); ax.legend(fontsize=8)
axes[0].set_ylabel("PC2")
plt.suptitle("Encoder output (CLS embedding), PCA projection", fontweight="bold")
plt.tight_layout()
plt.savefig(PLOT_DIR + "pca_nom_vs_flip.png", dpi=150, bbox_inches="tight")
print("Saved pca_nom_vs_flip.png")

In [ ]:
# Movement lines for one flavour at a time — set CLASS_TO_TRACE below.
CLASS_TO_TRACE = "b-jet"
trace_idx = CLASS_NAMES.index(CLASS_TO_TRACE)
m = y_sel == trace_idx
n_lines = min(300, m.sum())
line_sel = rng_plot.choice(np.where(m)[0], size=n_lines, replace=False)

fig, ax = plt.subplots(figsize=(7, 7))
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mm = y_sel == cls_idx
    ax.scatter(nom_2d[mm, 0], nom_2d[mm, 1], s=4, alpha=0.15,
               color=COLOURS[cls_name], label=f"{cls_name} (nom)", linewidths=0)
for i in line_sel:
    ax.plot([nom_2d[i, 0], flip_2d[i, 0]], [nom_2d[i, 1], flip_2d[i, 1]],
            color=COLOURS[CLASS_TO_TRACE], linewidth=0.6, alpha=0.6)
ax.scatter(flip_2d[line_sel, 0], flip_2d[line_sel, 1], s=10, marker="x",
           color=COLOURS[CLASS_TO_TRACE], label=f"{CLASS_TO_TRACE} (flip)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(fontsize=8)
ax.set_title(f"Nom -> flip movement for a sample of {CLASS_TO_TRACE} jets")
plt.tight_layout()
plt.savefig(PLOT_DIR + f"pca_movement_{CLASS_TO_TRACE.replace('-', '_')}.png", dpi=150, bbox_inches="tight")
print(f"Saved pca_movement_{CLASS_TO_TRACE.replace('-', '_')}.png")

## Plot: per-dimension embedding sensitivity

Which of the `d_model` latent dimensions does the flip perturb most, on average, for each flavour? Large bars for b and small bars for light in the same dimension is the signature of a well-separated encoder response.

In [ ]:
delta = h_flip - h_nom  # (N, d_model)
fig, ax = plt.subplots(figsize=(9, 5))
width = 0.8 / len(CLASS_NAMES)
x = np.arange(D_MODEL)
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = y == cls_idx
    mean_abs_delta = np.abs(delta[m]).mean(axis=0)
    ax.bar(x + cls_idx * width, mean_abs_delta, width=width,
           color=COLOURS[cls_name], label=cls_name)
ax.set_xlabel("Embedding dimension"); ax.set_ylabel("mean |h_flip - h_nom|")
ax.set_title("Per-dimension encoder-output sensitivity to flip, by flavour")
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR + "embedding_dim_sensitivity.png", dpi=150, bbox_inches="tight")
print("Saved embedding_dim_sensitivity.png")

## Plot: P(b) nominal vs flipped

Ties the embedding-space view back to the actual tagger output.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = y == cls_idx
    sel_m = rng_plot.choice(np.where(m)[0], size=min(N_PLOT_SAMPLE, m.sum()), replace=False)
    ax.scatter(p_nom[sel_m, 0], p_flip[sel_m, 0], s=4, alpha=0.3,
               color=COLOURS[cls_name], label=cls_name, linewidths=0)
ax.axline((0, 0), slope=1, color="black", linewidth=0.8, linestyle="--", label="y = x")
ax.set_xlabel("P(b) nominal"); ax.set_ylabel("P(b) flipped")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.set_title("P(b): nominal vs flipped")
plt.tight_layout()
plt.savefig(PLOT_DIR + "pb_nom_vs_flip.png", dpi=150, bbox_inches="tight")
print("Saved pb_nom_vs_flip.png")

## Reference: learned flip_scale

For context alongside the embedding-space plots above.

In [ ]:
learned_flip = model.flip_scale.detach().cpu().numpy()
flippable    = model.flip_mask.cpu().numpy()
for fld, s, f in zip(TRACK_FIELDS, learned_flip, flippable):
    tag = "" if f else "  (fixed, not trainable)"
    print(f"  {fld:45s} {s:+.3f}{tag}")